# ExpansionRx: multitask Chemprop vs fingerprint baseline

Train one directed message-passing neural network for all nine ADMET endpoints.

**Before starting:** select **Runtime → Change runtime type → CPU** if a GPU is unavailable. A T4 GPU also works; the notebook detects the allocated device automatically. Save a copy of this notebook to your Drive if you want to keep edits.

This notebook uses the same original training data, scaffold holdout and evaluation transformations as notebook 01. It never downloads the published challenge test labels. A GPU result is not precomputed or claimed.

We install the training stack in an isolated environment to avoid replacing Colab's preinstalled PyTorch/NumPy. The first setup downloads PyTorch and may take several minutes.

In [ ]:
import os, sys, subprocess, json
from pathlib import Path
REPO = Path(os.environ.get("EXPANSIONRX_REPO", "/content/expansionrx-chemprop"))
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/antoniaval/expansionrx-baselines.git", str(REPO)], check=True)
if not (REPO / "src/expansionrx/multitask.py").exists():
    raise RuntimeError("This checkout predates Chemprop. Use a fresh runtime or clone the current repository into a new folder.")
ENV = REPO / ".chemprop-env"
if not (ENV / "bin/python").exists():
    subprocess.run([sys.executable, "-m", "venv", str(ENV)], check=True)
PYTHON = str(ENV / "bin/python")
subprocess.run([PYTHON, "-m", "pip", "install", "-q", "-e", f"{REPO}[chemprop]"], check=True)
DEVICE = subprocess.check_output([PYTHON, "-c", "import torch; print('gpu' if torch.cuda.is_available() else 'cpu')"], text=True).strip()
print("Training device:", DEVICE.upper())
subprocess.run([PYTHON, "-c", "import torch, chemprop; print('PyTorch:', torch.__version__); print('Chemprop:', chemprop.__version__); print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"], check=True)


## Experiment settings and persistent storage

Default: 50 epochs, 300 hidden features, batch size 64. `SPLIT_SEED=42` preserves the baseline split; `MODEL_SEED` controls initialization and minibatch order. Changing only the model seed allows a fair repeated-initialization comparison.

Checkpoints and CSV logs are saved every epoch. Rerunning with identical settings resumes from the last completed epoch. Keep the same output path and package versions. Use a new run name after changing any model setting. For a quick installation check set `SMOKE_TEST=True`; its tiny model is **not** the main comparison.

Drive mounting is optional, but without it checkpoints disappear when the Colab VM is deleted.

In [ ]:
SAVE_TO_DRIVE = True
SMOKE_TEST = False
SPLIT_SEED = 42
MODEL_SEED = 42
EPOCHS = 2 if SMOKE_TEST else 50
HIDDEN_SIZE = 32 if SMOKE_TEST else 300
BATCH_SIZE = 64
DROPOUT = 0.1

IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS_ROOT = Path("/content/drive/MyDrive/ExpansionRx/runs")
else:
    RESULTS_ROOT = REPO / "runs"

RUN_NAME = f"chemprop_split{SPLIT_SEED}_seed{MODEL_SEED}_e{EPOCHS}_h{HIDDEN_SIZE}_b{BATCH_SIZE}_d{DROPOUT}"
OUTPUT = RESULTS_ROOT / RUN_NAME
RF_OUTPUT = RESULTS_ROOT / f"rf_seed{SPLIT_SEED}_trees200"
CACHE = REPO / "data"
print("Chemprop:", OUTPUT)
print("Fingerprint baseline:", RF_OUTPUT)


## Ensure the matched fingerprint baseline exists

If you saved notebook 01's default run in Drive, this reuses its completed endpoints. Otherwise it trains the same 200-tree baseline on CPU. This short step does not use the GPU. Different code/settings in an existing folder are rejected rather than silently mixed.

In [ ]:
subprocess.run([PYTHON, "-m", "expansionrx.baseline", "--output", str(RF_OUTPUT),
                "--cache", str(CACHE), "--seed", str(SPLIT_SEED), "--trees", "200", "--jobs", "2"], check=True)


## Train the multitask model

Missing labels remain missing and are masked by Chemprop. Each target is standardized using **training labels only**, after the organizer's log transformation. Checkpoint selection uses validation MSE in those standardized coordinates; final reporting unscales predictions and uses MAE/RAE in the same coordinates as notebook 01.

The lowest-validation-loss epoch is saved as `checkpoints/best.ckpt`; `last.ckpt` includes optimizer and scheduler state for resuming. The same holdout is used for epoch selection and comparison, so these are model-development scores, not an unbiased final test.

Leave the Colab session running. If interrupted, rerun setup, reconnect Drive and rerun with the same settings. Runtime depends on the allocated CPU or GPU; use the logs to measure it before scheduling multiple seeds.

In [ ]:
subprocess.run([
    PYTHON, "-m", "expansionrx.multitask",
    "--output", str(OUTPUT), "--cache", str(CACHE), "--baseline-dir", str(RF_OUTPUT),
    "--split-seed", str(SPLIT_SEED), "--seed", str(MODEL_SEED),
    "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE),
    "--hidden-size", str(HIDDEN_SIZE), "--dropout", str(DROPOUT),
    "--accelerator", DEVICE, "--progress"
], check=True)


## Compare the models

The comparison checks identical molecule/split assignments and per-endpoint label counts. **Lower RAE is better; negative delta means Chemprop wins.** A one-seed result does not establish superiority. If it is promising, keep the split fixed and repeat initialization seeds.

No claim is made that Chemprop must beat the forest; this experiment measures that.

In [ ]:
import pandas as pd
from IPython.display import display, Image
summary = json.loads((OUTPUT / "summary.json").read_text())
rf_summary = json.loads((RF_OUTPUT / "summary.json").read_text())
print("Chemprop macro RAE:", summary["macro_rae"])
print("Random forest macro RAE:", rf_summary["macro_rae"]["random_forest"])
comparison = pd.read_csv(OUTPUT / "comparison.csv")
display(comparison[["endpoint", "n_valid_chemprop", "rae_chemprop", "rae_rf", "delta_rae_chemprop_minus_rf"]])
display(Image(filename=str(OUTPUT / "comparison.png")))
print("Measured device:", summary["device"])
print("Peak allocated GPU memory (GB):", summary["max_gpu_allocated_gb"])


## Export results

The archive contains logs, predictions, scaler, configuration, split and comparison. Model checkpoints stay in Drive to avoid a large download. Keep **both** `best.ckpt` and `target_scaler.json` for later inference: network outputs are standardized values. Only load checkpoints you trust.

If a run runs out of GPU memory, start a new run with a smaller batch size. Do not change batch size inside an existing run folder.

In [ ]:
import zipfile
archive = OUTPUT.parent / f"{OUTPUT.name}_results.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for path in OUTPUT.rglob("*"):
        if path.is_file() and path.suffix in {".json", ".csv", ".png"}:
            bundle.write(path, arcname=str(path.relative_to(OUTPUT)))
print("Results:", archive)
if IN_COLAB:
    from google.colab import files
    files.download(str(archive))
